# Variables, Locals & Expressions

## What's covered

- **Input variables** — what they declare, where values come from, how to validate them
- **Type constraints** — primitives, collections, objects, the `any` escape hatch
- **`locals`** — for derived values and consistent naming
- **Outputs** — the cross-module interface and what `sensitive` actually does
- **Conditional expressions** and the **`for` expression** for transforming collections
- **Splat expressions** for collecting attributes across a `count` / `for_each` resource
- The **built-in function library** — strings, collections, files, encoding, and the ones you'll actually use


## The three input concepts

Every module — root or child — has three knobs for working with data:

| Block | Direction | Purpose |
|---|---|---|
| `variable` | input | A value passed *in*, set by CLI / tfvars / env / module call |
| `locals` | internal | A named value derived inside the module |
| `output` | output | A value exposed *out* — printed after apply, or returned to a parent module |

A useful mental model: variables and outputs are the module's **public interface**. Locals are its **internals**. Treat the interface as you would a function signature — minimal, well-named, documented.

Notebook 05 turns variables and outputs into the module interface explicitly. This notebook focuses on the language features that make them work — types, validation, expressions, and the function library.


## Input variables

A `variable` block declares an input. Four arguments matter:

```hcl
variable "instance_count" {
  type        = number
  default     = 3
  description = "Number of web instances to run."

  validation {
    condition     = var.instance_count >= 1 && var.instance_count <= 10
    error_message = "instance_count must be between 1 and 10."
  }
}
```

- **`type`** — the type constraint. If omitted, `any` (anything goes). Almost always supply a type.
- **`default`** — the value when no other source provides one. Without a default, the variable is *required*.
- **`description`** — shows up in `terraform plan` errors, in generated docs, and is read by anyone using the module. Write it.
- **`validation`** — a `condition` expression that must evaluate to `true`, and an `error_message` when it doesn't. Multiple `validation` blocks compose with logical AND.

Inside the module, the variable is referenced as `var.<name>`: `var.instance_count`.


### Where variable values come from

Six sources, in **precedence order** (later wins over earlier):

1. **Default** in the `variable` block.
2. **Environment variables** named `TF_VAR_<name>`. `TF_VAR_instance_count=5 terraform apply`.
3. **`terraform.tfvars`** in the working directory, if present.
4. **`*.auto.tfvars`** files in the working directory, in lexical order.
5. **`-var-file=path.tfvars`** flags, in the order given on the command line.
6. **`-var name=value`** flags, in the order given on the command line.

In practice, the common pattern:

```hcl
# terraform.tfvars — committed to git
region         = "us-east-1"
instance_count = 3

# secret.tfvars — gitignored, applied with -var-file=secret.tfvars
db_password = "..."
```

For CI: pass the variable file or `TF_VAR_*` env vars. Never inline secrets on the command line — they'll show up in shell history and process listings.


## Type constraints

Terraform's type system is simple but precise. Three primitive types, three collection types, two composite types, plus `any` and `null`.

### Primitives

`string`, `number`, `bool`. Conversion is automatic where it makes sense (string `"3"` to number `3`), but explicit is better than implicit.

### Collections

All elements must be the same type.

- **`list(T)`** — ordered, integer-indexed. `list(string)`, `list(number)`.
- **`set(T)`** — unordered, deduplicated. `set(string)`.
- **`map(T)`** — string-keyed, value-typed. `map(number)` maps strings to numbers.

```hcl
variable "subnet_cidrs" {
  type    = list(string)
  default = ["10.0.1.0/24", "10.0.2.0/24", "10.0.3.0/24"]
}
```

### Structural types

Elements can be different types.

- **`object({ ... })`** — fixed set of named attributes with specific types.
- **`tuple([ ... ])`** — fixed-length, position-typed.

```hcl
variable "subnet" {
  type = object({
    cidr_block = string
    az         = string
    public     = bool
    tags       = optional(map(string), {})
  })
}
```

The **`optional()`** wrapper marks an attribute as optional with a default value (Terraform 1.3+). Without it, every attribute in an `object` is required.

### `any`

`type = any` accepts anything. Useful for module interfaces that genuinely take arbitrary structures, but most of the time it's a lazy alternative to writing a real type constraint. Prefer concrete types.

### `null`

A literal value meaning "no value." Most arguments accept `null` as "skip this attribute"; the resource behaves as if the argument wasn't set. The `null_resource` of yesteryear is unrelated — it's a (now-deprecated) resource type, not the type system's null.


## `locals` — derived values

A `locals` block declares one or more named values that are computed once and reused. Use it for:

- **Naming consistency.** Generate every resource's name from one local.
- **Repeated expressions.** Don't write the same `merge(var.tags, ...)` three times — compute it once.
- **Readability.** Give a long expression a name that says what it represents.

```hcl
locals {
  env         = "prod"
  region      = "us-east-1"
  name_prefix = "${local.env}-${local.region}"

  common_tags = {
    Environment = local.env
    Region      = local.region
    ManagedBy   = "terraform"
  }

  # A derived list — three subnets named consistently.
  subnets = {
    a = { cidr = "10.0.1.0/24", az = "us-east-1a" }
    b = { cidr = "10.0.2.0/24", az = "us-east-1b" }
    c = { cidr = "10.0.3.0/24", az = "us-east-1c" }
  }
}

resource "aws_s3_bucket" "logs" {
  bucket = "${local.name_prefix}-logs"
  tags   = local.common_tags
}
```

Locals are referenced as `local.<name>`. Multiple `locals` blocks per file are allowed — all merged into one namespace.

**A practical pattern.** A single `locals.tf` file at the top of the module holds derived names, tag maps, and computed inputs. It functions as the "where did this name come from?" lookup for the whole module.


## Outputs

An `output` block exposes a value. Two audiences:

- **Humans** — the value is printed after `terraform apply`, queryable via `terraform output`.
- **Other modules** — when this module is used as a child, its outputs are visible as `module.<name>.<output>` (notebook 05).

```hcl
output "bucket_arn" {
  value       = aws_s3_bucket.logs.arn
  description = "ARN of the logs bucket."
}

output "db_password" {
  value     = aws_db_instance.main.password
  sensitive = true
}

output "instance_ids" {
  value = [for i in aws_instance.web : i.id]
}
```

The arguments:

- **`value`** — the expression to expose. Anything that evaluates: a single attribute, a literal, a `for` expression, a function call.
- **`description`** — surfaced in `terraform output` and in generated docs.
- **`sensitive = true`** — Terraform masks the value in plan and apply output (`<sensitive>` instead of the literal). The value is *still in state* in plain text — see notebook 03. Use this for output convenience, not for security.

**`terraform output`** at the command line:

```bash
$ terraform output
bucket_arn = "arn:aws:s3:::myorg-prod-logs"

$ terraform output -raw bucket_arn      # no quotes, useful in shell pipelines
arn:aws:s3:::myorg-prod-logs

$ terraform output -json                # structured for tools
{
  "bucket_arn": {
    "value": "arn:aws:s3:::myorg-prod-logs",
    "type": "string"
  }
}
```


## Conditional expressions

The ternary `condition ? true_value : false_value` is the only conditional form in HCL.

```hcl
locals {
  instance_type = var.env == "prod" ? "t3.large" : "t3.micro"
  backup_window = var.env == "prod" ? "02:00-03:00" : null
  replica_count = var.high_availability ? 3 : 1
}
```

A few patterns worth knowing:

**Toggle a resource on or off:**

```hcl
resource "aws_cloudwatch_metric_alarm" "high_latency" {
  count = var.enable_alarms ? 1 : 0
  # ...
}
```

`count = 0` means the resource doesn't exist; `count = 1` means it does. Reference as `aws_cloudwatch_metric_alarm.high_latency[0]`.

**Null to skip an argument:**

```hcl
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = "t3.micro"
  key_name      = var.ssh_key_name != "" ? var.ssh_key_name : null
}
```

When `var.ssh_key_name` is empty, `null` means "don't set `key_name`" — the resource behaves as if the argument wasn't written.

**Mixed types are an error.** Both branches of `? :` must produce the same type. `var.env == "prod" ? "large" : 1` errors at plan time.


## `for` expressions — transforming collections

The `for` expression is Terraform's list comprehension. Two forms — one produces a list, one produces a map.

### List form — `[for x in list : expr]`

```hcl
locals {
  subnet_cidrs = ["10.0.1.0/24", "10.0.2.0/24", "10.0.3.0/24"]
  subnet_ids   = [for c in local.subnet_cidrs : cidrhost(c, 0)]
}
# subnet_ids = ["10.0.1.0", "10.0.2.0", "10.0.3.0"]
```

### Map form — `{for k, v in map : new_k => new_v}`

```hcl
locals {
  servers = {
    web = "t3.micro"
    api = "t3.small"
    db  = "r5.large"
  }
  # Build a map of name -> uppercase name.
  upper_names = { for k, _ in local.servers : k => upper(k) }
}
# upper_names = { web = "WEB", api = "API", db = "DB" }
```

### Filtering with `if`

```hcl
locals {
  prod_subnets = [for s in var.subnets : s if s.env == "prod"]
}
```

Only includes elements where the `if` clause is true. Powerful for picking out a slice of a config.

### A realistic example — building tags

```hcl
locals {
  base_tags = {
    Environment = "prod"
    Team        = "platform"
  }

  # Per-resource extra tags.
  per_resource = {
    "web"  = { Tier = "frontend" }
    "api"  = { Tier = "backend" }
    "db"   = { Tier = "data" }
  }

  # Combine base tags with per-resource extras.
  all_tags = {
    for name, extra in local.per_resource :
    name => merge(local.base_tags, extra, { Name = name })
  }
}
```

`for` is the workhorse of nontrivial HCL. Once you read it fluently, most "how do I express this?" questions answer themselves.


## Splat expressions

When a resource has many instances (via `count` or `for_each`), you often want a list of one attribute across all of them. **Splat** is the shortcut.

```hcl
resource "aws_instance" "web" {
  count         = 3
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
}

# Three equivalent ways to get the list of IDs:
output "ids_splat"    { value = aws_instance.web[*].id }
output "ids_for"      { value = [for i in aws_instance.web : i.id] }
output "ids_legacy"   { value = aws_instance.web.*.id }   # older syntax
```

The `[*]` splat works on `count`-resources and `for_each`-resources alike (returning a list of attribute values).

**Limitation:** splat can't chain through nested attributes that are themselves lists. For complex traversal, `for` is more flexible.

```hcl
# Splat — fine for one level deep.
output "subnet_arns" { value = aws_subnet.public[*].arn }

# For complex shapes, use for.
output "subnet_cidrs_by_az" {
  value = { for s in aws_subnet.public : s.availability_zone => s.cidr_block }
}
```


## Built-in functions — the ones you'll use

Terraform ships ~100 built-in functions. You'll reach for maybe 20 regularly. Categorized:

### String

- **`format(spec, args...)`** — printf-style. `format("hello-%s-%02d", env, i)`.
- **`replace(s, search, replace)`** — substring or regex replacement.
- **`lower(s)` / `upper(s)` / `title(s)`** — case manipulation.
- **`trimspace(s)` / `trimprefix(s, p)` / `trimsuffix(s, s)`** — trimming.
- **`join(sep, list)` / `split(sep, s)`** — list/string round-trip.

### Collection

- **`merge(m1, m2, ...)`** — combine maps. Later wins on conflicts. The standard tag-merging idiom.
- **`concat(l1, l2, ...)`** — combine lists.
- **`keys(m)` / `values(m)`** — extract from a map.
- **`length(c)`** — works on string, list, set, map.
- **`contains(list, value)` / `lookup(map, key, default)`** — membership and safe map access.
- **`tolist(s)` / `toset(l)` / `tomap(o)`** — explicit conversion.
- **`distinct(list)`** — deduplicate.
- **`flatten(list_of_lists)`** — one level of unnesting. Handy for transforming `for_each` data.

### File / encoding

- **`file(path)`** — read a file's contents as a string. The path is resolved relative to the module.
- **`templatefile(path, vars)`** — render a `.tpl` template file with a vars map. The modern alternative to inline heredocs.
- **`jsonencode(v)` / `jsondecode(s)`** — round-trip with JSON. Used heavily for IAM policy documents.
- **`yamlencode(v)` / `yamldecode(s)`** — same, for YAML.
- **`base64encode(s)` / `base64decode(s)`** — for cloud-init user data and Lambda zip files.

### Numeric / time

- **`max(a, b, ...)` / `min(a, b, ...)`** — usual.
- **`floor(n)` / `ceil(n)`** — rounding.
- **`timestamp()`** — current ISO 8601 time. *Be careful* — using it in a resource argument makes the resource update on every plan.
- **`formatdate(spec, t)`** — format a timestamp.

### Cryptographic / hashing

- **`md5(s)` / `sha256(s)`** — hashes.
- **`filemd5(path)` / `filesha256(path)`** — hash of a file. Common in Lambda deployment for `source_code_hash`.
- **`uuid()`** — generate a UUID. Same caveat as `timestamp()` — produces a new value every plan; use `random_id` for stable identifiers.

### IP / CIDR

- **`cidrsubnet(prefix, newbits, netnum)`** — carve a subnet out of a CIDR. Indispensable for VPC layouts.
- **`cidrhost(prefix, hostnum)`** — host address from a CIDR.

### A short example

```hcl
locals {
  # Read a policy template, fill in variables, json-encode (validates the JSON).
  bucket_policy = jsonencode(jsondecode(templatefile("${path.module}/policy.json.tpl", {
    bucket_arn = aws_s3_bucket.logs.arn
    account_id = data.aws_caller_identity.current.account_id
  })))

  # Carve 4 /24s out of the VPC CIDR.
  subnet_cidrs = [for i in range(4) : cidrsubnet(var.vpc_cidr, 8, i)]

  # File hash for Lambda redeployment.
  lambda_hash = filebase64sha256("${path.module}/lambda.zip")
}
```

The full function reference lives at `terraform.io/language/functions`. Bookmark it; you'll consult it often.


## Forward

Notebook five turns to **Modules & Composition** — how to package configuration as reusable units. Root vs child modules. The conventional file structure (`main.tf`, `variables.tf`, `outputs.tf`, `versions.tf`). The public registry. Versioning. The composition patterns — wrapping, layering, factory — that show up in real codebases. And the equally important question of when *not* to make a module yet. Variables and outputs from this notebook become the module's interface; locals stay its internals.
